In [10]:
import ee
import geemap
import geopandas as gpd
import json
import requests
import zipfile
import os
from io import BytesIO
from shapely.geometry import Point

def baixar_e_extrair_zip(url_base, ano):
    # Nome do arquivo ZIP
    nome_arquivo_zip = f"focos_br_ref_{ano}.zip"
    url = f"{url_base}/{nome_arquivo_zip}"

    # Verifica ou cria o diretório "shp"
    pasta_shp = os.path.join(os.getcwd(), 'shp')
    if not os.path.exists(pasta_shp):
        os.makedirs(pasta_shp)

    # Caminho completo para o arquivo ZIP na pasta "shp"
    caminho_zip = os.path.join(pasta_shp, nome_arquivo_zip)

    # Faz o download do arquivo ZIP
    response = requests.get(url)
    if response.status_code != 200:
        raise FileNotFoundError(f"Erro ao baixar o arquivo ZIP: {url}")

    # Salva o arquivo ZIP na pasta "shp"
    with open(caminho_zip, 'wb') as f:
        f.write(response.content)

    if not os.path.exists(caminho_zip):
        raise FileNotFoundError(f"Arquivo ZIP não foi baixado: {caminho_zip}")

    # Extrai o conteúdo do ZIP
    with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
        arquivos_no_zip = zip_ref.namelist()
        print("Arquivos no ZIP:", arquivos_no_zip)

        # Procura pelo arquivo CSV dentro do ZIP
        arquivo_csv = next((arquivo for arquivo in arquivos_no_zip 
                            if arquivo.endswith('.csv')), None)

        if not arquivo_csv:
            raise FileNotFoundError(f"Nenhum arquivo CSV encontrado no ZIP: {caminho_zip}")

        # Extrai o arquivo CSV diretamente para a pasta "shp"
        zip_ref.extract(arquivo_csv, pasta_shp)

    # Caminho completo para o arquivo CSV extraído
    caminho_csv = os.path.join(pasta_shp, arquivo_csv)

    print(f"Arquivo CSV extraído para a pasta 'shp': {caminho_csv}")
    return caminho_csv

ee.Initialize()

def obtem_ano(ano):
    data_inicio = f"{ano}-07-01"
    data_fim = f"{ano}-10-31"
    
    # Carrega o shapefile do Parque Nacional da Serra da Canastra
    pnsc = gpd.read_file(r"G:\Meu Drive\@EquipeGEO\zz.Bases\ICMBio\UC_Fed_nov_2020.shp")
    pnsc = pnsc.loc[pnsc["nome"] == "PARQUE NACIONAL DA SERRA DA CANASTRA"]
    pnsc_geojson = pnsc.to_json()

    # Converte o shapefile para um FeatureCollection do Earth Engine
    pnsc_ee = ee.FeatureCollection(json.loads(pnsc_geojson))

    # Carrega os dados de focos de queimadas
    gdf = gpd.read_file(r"C:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_"+str(ano)+".csv")
    gdf['geometry'] = gpd.points_from_xy(gdf['lon'], gdf['lat'])
    gdf = gpd.GeoDataFrame(gdf, geometry='geometry')

    # Verifica e converte colunas do tipo bytes para strings
    for col in gdf.columns:
        if gdf[col].dtype == object:  # Apenas verifica colunas do tipo 'object'
            gdf[col] = gdf[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

    # Filtra os focos de queimadas para o estado de Minas Gerais e dentro do período de interesse
    focos = gdf.loc[(gdf['estado'] == 'MINAS GERAIS') & (gdf['data_pas'] >= data_inicio) & (gdf['data_pas'] <= data_fim)]
    focos_geojson = focos.to_json()

    # Converte os focos de queimadas para um FeatureCollection do Earth Engine
    focos_ee = ee.FeatureCollection(json.loads(focos_geojson))

    # Filtra os focos de queimadas para aqueles que estão dentro do parque
    focos_ee = focos_ee.filterBounds(pnsc_ee.geometry())

    # Função para selecionar a coleção Landsat correta com base no ano
    def selecionarColecaoLandsat(ano):
        if 1984 <= ano <= 1999:
            return "LANDSAT/LT05/C02/T1_L2"
        elif 1999 <= ano <= 2012:
            return "LANDSAT/LE07/C02/T1_L2"
        elif ano >= 2013:
            return "LANDSAT/LC08/C02/T1_L2"
        else:
            raise ValueError("Ano fora do intervalo disponível para coleções Landsat")

    # Função para mascarar nuvens e sombras
    def maskLandsat(image):
        qa = image.select('QA_PIXEL')
        cloud = qa.bitwiseAnd(1 << 3).eq(0)
        shadow = qa.bitwiseAnd(1 << 5).eq(0)
        mask = cloud.And(shadow)
        return image.updateMask(mask)

    # Função para mascarar água usando NDWI
    def maskWater(image):
        ndwi = image.normalizedDifference(["SR_B3", "SR_B5"])  # B3 = Verde, B5 = Infravermelho Próximo (Landsat 8)
        water_mask = ndwi.lt(0)  # Mantém apenas áreas onde NDWI < 0 (não água)
        return image.updateMask(water_mask)

    # Seleciona a coleção Landsat correta
    colecao = selecionarColecaoLandsat(ano)

    # Filtra e aplica a máscara na coleção de imagens
    dataset = ee.ImageCollection(colecao) \
        .filterDate(data_inicio, data_fim) \
        .filterBounds(pnsc_ee.geometry()) \
        .map(maskLandsat) \
        .map(maskWater)  # Aplica a máscara de água

    # Calcula o NBR (Normalized Burn Ratio)
    if ano <= 2012:
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B4", "SR_B7"]).rename("NBR"))
    else:
        nbr = dataset.map(lambda image: image.normalizedDifference(["SR_B5", "SR_B7"]).rename("NBR"))

    nbrMin = nbr.min()

    # Identifica áreas com cicatrizes de incêndios (NBR entre -1 e 0)
    cicatrizes = nbrMin.updateMask(nbrMin.gte(-1).And(nbrMin.lte(0)))

    # Filtra as cicatrizes para aquelas que estão dentro do parque
    cicatrizes = cicatrizes.clip(pnsc_ee.geometry())

    # Sobrepor os focos de queimadas com as áreas de cicatrizes
    focos_cicatrizes = focos_ee.filterBounds(cicatrizes.geometry())

    # Conta quantos focos estão sobrepostos às cicatrizes
    quantidade_focos_cicatrizes = focos_cicatrizes.size().getInfo()

    print(f"Quantidade de focos de queimadas sobrepostos a cicatrizes de incêndios dentro do parque: {quantidade_focos_cicatrizes}")

    # Obtém o centróide do parque para centralizar o mapa
    centroid = pnsc_ee.geometry().centroid()

    # Cria o mapa interativo
    Map = geemap.Map(center=(centroid.coordinates().get(1).getInfo(), centroid.coordinates().get(0).getInfo()), zoom=10)

    # Adiciona as camadas ao mapa
    Map.addLayer(cicatrizes, {"min": -1, "max": 0, "palette": ["orange"]}, "Cicatrizes de Incêndios (NBR entre -1 e 0)")
    Map.addLayer(ee.Image().paint(pnsc_ee, 0, 2), {}, "Limite do PARNA Serra da Canastra")
    Map.addLayer(focos_cicatrizes, {"color": "red"}, "Focos de Calor sobre Cicatrizes")

    return Map

In [3]:
for ano in range(2003, 2025):
    url_base = 'https://dataserver-coids.inpe.br/queimadas/queimadas/focos/csv/anual/Brasil_sat_ref/'
    arquivo_csv = baixar_e_extrair_zip(url_base, ano)

Arquivos no ZIP: ['focos_br_ref_2003.csv']
Arquivo CSV extraído para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2003.csv
Arquivos no ZIP: ['focos_br_ref_2004.csv']
Arquivo CSV extraído para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2004.csv
Arquivos no ZIP: ['focos_br_ref_2005.csv']
Arquivo CSV extraído para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2005.csv
Arquivos no ZIP: ['focos_br_ref_2006.csv']
Arquivo CSV extraído para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2006.csv
Arquivos no ZIP: ['focos_br_ref_2007.csv']
Arquivo CSV extraído para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref

In [11]:
# Testando para o ano de 2022
mapa = obtem_ano(2022)
mapa

Quantidade de focos de queimadas sobrepostos a cicatrizes de incêndios dentro do parque: 21


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [12]:
# Testando para o ano de 2021
mapa = obtem_ano(2021)
mapa

Quantidade de focos de queimadas sobrepostos a cicatrizes de incêndios dentro do parque: 33


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [46]:
# Testando para o ano de 2020
mapa = obtem_ano(2020)
mapa

Arquivos no ZIP: ['focos_br_ref_2020.csv']
Lendo focos_br_ref_2020.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [47]:
# Testando para o ano de 2019
mapa = obtem_ano(2019)
mapa

Arquivos no ZIP: ['focos_br_ref_2019.csv']
Lendo focos_br_ref_2019.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [48]:
# Testando para o ano de 2018
mapa = obtem_ano(2018)
mapa

Arquivos no ZIP: ['focos_br_ref_2018.csv']
Lendo focos_br_ref_2018.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [49]:
# Testando para o ano de 2017
mapa = obtem_ano(2017)
mapa

Arquivos no ZIP: ['focos_br_ref_2017.csv']
Lendo focos_br_ref_2017.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [50]:
# Testando para o ano de 2016
mapa = obtem_ano(2016)
mapa

Arquivos no ZIP: ['focos_br_ref_2016.csv']
Lendo focos_br_ref_2016.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [51]:
# Testando para o ano de 2015
mapa = obtem_ano(2015)
mapa

Arquivos no ZIP: ['focos_br_ref_2015.csv']
Lendo focos_br_ref_2015.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [52]:
# Testando para o ano de 2014
mapa = obtem_ano(2014)
mapa

Arquivos no ZIP: ['focos_br_ref_2014.csv']
Lendo focos_br_ref_2014.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [53]:
# Testando para o ano de 2013
mapa = obtem_ano(2013)
mapa

Arquivos no ZIP: ['focos_br_ref_2013.csv']
Lendo focos_br_ref_2013.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [54]:
# Testando para o ano de 2012
mapa = obtem_ano(2012)
mapa

Arquivos no ZIP: ['focos_br_ref_2012.csv']
Lendo focos_br_ref_2012.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [55]:
# Testando para o ano de 2011
mapa = obtem_ano(2011)
mapa

Arquivos no ZIP: ['focos_br_ref_2011.csv']
Lendo focos_br_ref_2011.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [56]:
# Testando para o ano de 2010
mapa = obtem_ano(2010)
mapa

Arquivos no ZIP: ['focos_br_ref_2010.csv']
Lendo focos_br_ref_2010.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [58]:
# Testando para o ano de 2009
mapa = obtem_ano(2009)
mapa

Arquivos no ZIP: ['focos_br_ref_2009.csv']
Arquivo CSV movido para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2009.csv
Lendo c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2009.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [59]:
# Testando para o ano de 2008
mapa = obtem_ano(2008)
mapa

Arquivos no ZIP: ['focos_br_ref_2008.csv']
Arquivo CSV movido para a pasta 'shp': c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2008.csv
Lendo c:\Users\Diego\OneDrive - Universidade de Vassouras\Diego\Dissertação 2022\Change_Detection\shp\focos_br_ref_2008.csv com GeoPandas...


Map(center=[-20.332779749369294, -46.58433098034936], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
# Testando para o ano de 2007
mapa = obtem_ano(2007)
mapa

In [ ]:
# Testando para o ano de 2006
mapa = obtem_ano(2006)
mapa

In [ ]:
# Testando para o ano de 2005
mapa = obtem_ano(2005)
mapa

In [ ]:
# Testando para o ano de 2004
mapa = obtem_ano(2004)
mapa

In [ ]:
# Testando para o ano de 2003
mapa = obtem_ano(2003)
mapa